# Cross-Domain Sentiment Analysis: Mitigating Domain Shift

## 1. Project Overview
This project tackles **Domain Shift** in Sentiment Analysis. Sentiment models trained heavily on one type of text (e.g., formal reviews or short tweets) often fail when applied to a new domain (e.g., private chats or student emails). 

Our goal is to build a robust ensemble model that can generalize across diverse domains, specifically focusing on the shift from general social media (Twitter) to student-life contexts (Gmail, WhatsApp, and Google Play Store App Reviews).

### Key Objectives:
- **Training Domain:** General sentiment from social media (Twitter).
- **Target Domains (Domain Shift):** 
  1. Private WhatsApp messages (Student slang/chat).
  2. Student-related Gmail threads (Formal/Transactional).
  3. Google App Reviews (Product Feedback - Manually Labeled).
- **The Ensemble Approach:** Stacking 10 diverse models (RNNs + Transformers) to ensure high predictive stability across all domains.

## 2. Importing the necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import random
import gc
import zipfile
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SpatialDropout1D, Conv1D, GlobalMaxPooling1D, LSTM, Bidirectional, GRU, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

import xgboost as xgb
from lime.lime_text import LimeTextExplainer
from matplotlib.patches import FancyBboxPatch

sns.set_theme(style="whitegrid", palette="muted")
gpus = tf.config.list_physical_devices('GPU')
if gpus: 
    try: tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=4096)])
    except RuntimeError as e: print(e)
SEED = 42
def seed_everything(seed=42):
    random.seed(seed) 
    os.environ['PYTHONHASHSEED'] = str(seed) 
    np.random.seed(seed)
    tf.random.set_seed(seed) 
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)
seed_everything(SEED); DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Data Loading

This section handles the loading of data at different stages of the pipeline: from raw source files for cleaning, to processed datasets for model training and evaluation.

In [ ]:
# Training data
df_sentiment = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/sentimentdataset.csv')
df_tweets = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/Tweets.csv')
df_twitterdataset = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/twitterdataset.csv', header=None)

df_twitterdataset.columns  = ["id",'topic',"sentiment","text"]


# Gmail and whatsapp data for testing the models
df_gmail_raw = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/gmail_raw.csv")
df_whatsapp_raw = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/whatsapp_raw.csv")
df_app_reviews_raw = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/app_reviews_raw.csv")


# Load clean Twitter training data
train_df = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/processed_training_dataset.csv').dropna()
val_df = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/processed_validation_datset.csv').dropna()

# Combine training and validation for the larger ensemble training pool
train_df = pd.concat([train_df, val_df]).reset_index(drop=True)

# Load ready-to-predict test datasets
df_gmail_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/gmail_test_data.csv")
df_whatsapp_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/whatsapp_test_data.csv")
df_app_reviews_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/labelled_app_reviews_test.csv")


print("Done with data loading...")

## 4. Exploratory Data Analysis (EDA)

## 4.1 EDA for Training Dataset (Raw)

### Shape of the data

In [ ]:
print("=== RAW TRAINIGN AND VALIDATION DATA LOADED ===")
print(f"sentimentdataset.csv  → {df_sentiment.shape[0]} rows, {df_sentiment.shape[1]} columns")
print(f"Tweets.csv            → {df_tweets.shape[0]} rows, {df_tweets.shape[1]} columns")
print(f"twitterdataset.csv    → {df_twitterdataset.shape[0]} rows, {df_twitterdataset.shape[1]} columns")


**Insight**: We loaded three datasets for training. `sentimentdataset.csv` (732 rows) and `Tweets.csv` (14,640 rows) have many extra features (15 columns). The biggest dataset, `twitterdataset.csv`, has over 74,000 rows but only 4 columns. We have nearly 90,000 examples to train our model on, but we need to drop the extra columns and just keep the text and sentiment labels to merge them easily.

### Columns available for the three datasets

### Viewing the first few rows

In [ ]:
display("--- Sentiment Dataset ---")
display(df_sentiment.head(3))

display("--- Tweets Dataset ---")
display(df_tweets.head(3))

display("--- Twitter Dataset ---")
display(df_twitterdataset.head(3))

**Insight**: Looking at the first few rows shows that the column names don't match. For example, the labels are called `Sentiment` in one file but `airline_sentiment` in another. We need to rename these columns so they match before we can combine the datasets.

### Dataset Summaries (Info and Desccribe)

In [ ]:
print("=== INFO: sentimentdataset.csv ===")
df_sentiment.info()
print("\n=== DESCRIBE: sentimentdataset.csv ===")
display(df_sentiment.describe(include='all'))

print("\n\n=== INFO: Tweets.csv ===")
df_tweets.info()
print("\n=== DESCRIBE: Tweets.csv ===")
display(df_tweets.describe(include='all'))

print("\n\n=== INFO: twitterdataset.csv ===")
df_twitterdataset.info()
print("\n=== DESCRIBE: twitterdataset.csv ===")
display(df_twitterdataset.describe(include='all'))

**Insight**: The data summaries show that the text features are mostly complete, but there are a lot of extra columns (like dates or user IDs) that our NLP model doesn't need. We will need to remove these extra features later.

### Columns avaialble in the training dataset

In [ ]:

print("sentimentdataset.csv columns:", df_sentiment.columns.tolist())
print("Tweets.csv columns:", df_tweets.columns.tolist())
print("twitterdataset.csv columns:", df_twitterdataset.columns.tolist())



**Insight**: By looking at the exact column names, we know exactly which ones to keep. For our ML model, we only care about the input feature (the text) and the target label (the sentiment). We will drop everything else.

### Checking for null values

In [ ]:
print("\n=== NULL VALUES CHECK ===")
print("df_sentiment nulls:\n", df_sentiment.isnull().sum())
print()
print("df_tweets nulls:\n", df_tweets.isnull().sum())
print()
print("df_twitterdataset nulls:\n", df_twitterdataset.isnull().sum())
print()


### Checking for rows where the text is just whitespace 

In [ ]:
df_sentiment = df_sentiment[df_sentiment['Text'].str.strip().str.len() > 0]
df_tweets = df_tweets[df_tweets['text'].str.strip().str.len() > 0]
df_twitterdataset = df_twitterdataset[df_twitterdataset['text'].str.strip().str.len() > 0]

### Checking the Class Distribution

In [ ]:
print("sentimentdataset.csv distribution")
print(df_sentiment.Sentiment.value_counts())
print("")
print("tweets.csv distribution")
print(df_tweets.airline_sentiment.value_counts())
print("")
print("twitterdataset.csv distribution")
print(df_twitterdataset.sentiment.value_counts())

**Insight**: The class distribution shows that our data is imbalanced. For example, some datasets have way more 'Positive' or 'Negative' labels than others. Also, we have messy labels like 'Irrelevant'. To train the model properly, we need to map all these different labels into just three clean categories: Positive, Negative, and Neutral.

## 4.2 EDA for the Test Data (Raw)

### Viewing first few rows of the data

In [ ]:
display("--- Gmail (Raw) ---")
display(df_gmail_raw.head(3))
display("--- WhatsApp (Raw) ---")
display(df_whatsapp_raw.head(3))
display("--- App Reviews (Raw) ---")
display(df_app_reviews_raw.head(3))

### Dataset Summaries ( Describe and Info)

In [ ]:

print("=== INFO: gmail_raw.csv ===")
df_gmail_raw.info()
print("\n=== DESCRIBE: gmailraw.csv ===")
display(df_gmail_raw.describe(include='all'))

print("\n\n=== INFO: whatsapp_raw.csv ===")
df_whatsapp_raw.info()
print("\n=== DESCRIBE: whatsapp_raw.csv ===")
display(df_whatsapp_raw.describe(include='all'))

print("\n\n=== INFO: app_reviews_raw.csv ===")
df_app_reviews_raw.info()
print("\n=== DESCRIBE: app_reviews_raw.csv ===")
display(df_app_reviews_raw.describe(include='all'))

### Checking for null values 

In [ ]:
print("\n=== TEST DATA NULL VALUES CHECK ===")
print("df_gmail nulls:\n", df_gmail_raw.isnull().sum())
print()
print("df_whatsapp nulls:\n", df_whatsapp_raw.isnull().sum())
print()
print("df_app_reviews nulls:\n", df_app_reviews_raw.isnull().sum())
print()

### Checking the class Distribution 

In [ ]:
print("df_gmail distribution")
print(df_gmail_raw.sentiment.value_counts(dropna=False))
print("")
print("df_whatsapp distribution")
print(df_whatsapp_raw.sentiment.value_counts(dropna=False))
print("")
print("df_app_reviews distribution")
print(df_app_reviews_raw.sentiment.value_counts(dropna=False))

### Visualising the class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Common colors for consistency
sentiment_colors = {
    'positive': '#4CAF50',
    'neutral': '#2196F3',
    'negative': '#F44336'
}

# Helper to plot mapped colors or fallback to a default
def get_colors(index):
    return [sentiment_colors.get(str(i).lower(), '#9E9E9E') for i in index]

# Dataset 1: Gmail
gmail_counts = df_gmail_raw['sentiment'].value_counts(dropna=False)
gmail_counts.plot(
    kind='bar', ax=axes[0], color=get_colors(gmail_counts.index), edgecolor='black', width=0.6
)
axes[0].set_title('Gmail — Label Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Number of Rows')
axes[0].tick_params(axis='x', rotation=45)
for p in axes[0].patches:
    axes[0].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )


# Dataset 2: WhatsApp
whatsapp_counts = df_whatsapp_raw['sentiment'].value_counts(dropna=False)
whatsapp_counts.plot(
    kind='bar', ax=axes[1], color=get_colors(whatsapp_counts.index), edgecolor='black', width=0.6
)
axes[1].set_title('WhatsApp — Label Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment Class')
axes[1].set_ylabel('Number of Rows')
axes[1].tick_params(axis='x', rotation=45)
for p in axes[1].patches:
    axes[1].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )


# Dataset 3: App Reviews
app_reviews_counts = df_app_reviews_raw['sentiment'].value_counts(dropna=False)
app_reviews_counts.plot(
    kind='bar', ax=axes[2], color=get_colors(app_reviews_counts.index), edgecolor='black', width=0.6
)
axes[2].set_title('App Reviews — Label Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Sentiment Class')
axes[2].set_ylabel('Number of Rows')
axes[2].tick_params(axis='x', rotation=45)
for p in axes[2].patches:
    axes[2].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )


plt.suptitle('Class Distribution in Test Datasets (Pre-Processing)', fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig('viz_test_distribution_before_merge.png', dpi=150, bbox_inches='tight')
plt.show()

# still need running to be sure 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sentiment_colors = {'positive': '#4CAF50', 'neutral': '#2196F3', 'negative': '#F44336'}
def get_colors(index):
    return [sentiment_colors.get(str(i).lower(), '#9E9E9E') for i in index]

df_gmail_raw['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=get_colors(df_gmail_raw['sentiment'].value_counts().index), edgecolor='black')
axes[0].set_title('Gmail Labels')

df_whatsapp_raw['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=get_colors(df_whatsapp_raw['sentiment'].value_counts().index), edgecolor='black')
axes[1].set_title('WhatsApp Labels')

if 'score' in df_app_reviews_raw.columns:
    df_app_reviews_raw['score'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='skyblue', edgecolor='black')
    axes[2].set_title('App Reviews Scores')

plt.suptitle('Class Distribution in Test Datasets (Pre-Processing)')
plt.show()

## 5. Data Cleaning And Preprocessing

In [ ]:
# Maps any sentiment label to one of three standard classes: Positive, Negative, or Neutral and Returns None for labels we cannot confidently map.

def normalize_label(label):
    
    label = str(label).strip().lower()
 
    positive_labels = [
        'positive', 'joy', 'happiness', 'happy', 'love', 'excitement',
        'excited', 'contentment', 'content', 'gratitude', 'grateful',
        'serenity', 'serene', 'hopeful', 'hope', 'awe', 'pride',
        'acceptance', 'relief', 'mischievous', 'enthusiasm'
    ]
 
    negative_labels = [
        'negative', 'anger', 'angry', 'fear', 'sadness', 'sad',
        'disgust', 'disgusted', 'grief', 'despair', 'loneliness',
        'lonely', 'embarrassed', 'embarrassment', 'hate', 'bad',
        'nostalgia', 'confusion', 'confused', 'anxiety', 'anxious'
    ]
 
    neutral_labels = [
        'neutral', 'curiosity', 'curious', 'surprise', 'surprised'
    ]
 
    if label in positive_labels:
        return 'Positive'
    elif label in negative_labels:
        return 'Negative'
    elif label in neutral_labels:
        return 'Neutral'
    else:
        # Any label we cannot confidently classify is dropped
        return None
 

## 5.1 Training Data Preprocessing and Cleaning

### Dropping useless Columns for the trainiing datasets

In [ ]:
df_sentiment_raw = df_sentiment[['Text', 'Sentiment']]
df_tweets_raw = df_tweets[['text', 'airline_sentiment']]
df_twitterdataset_raw = df_twitterdataset[['text', 'sentiment']]

### Renaming the training dataset columns to a consistent standard 

In [ ]:
df_sentiment_raw = df_sentiment.rename(columns={'Text': 'text', 'Sentiment': 'sentiment'})
df_tweets_raw = df_tweets.rename(columns={'airline_sentiment': 'sentiment'})
df_twitterdataset_raw = df_twitterdataset.rename(columns={'sentiment': 'sentiment'})

### Viewing the available columns

In [ ]:
print("\n=== AFTER DROPPING USELESS COLUMNS ===")
print(f"df_sentiment columns: {df_sentiment_raw.columns.tolist()}")
print(f"df_tweets columns: {df_tweets_raw.columns.tolist()}")
print(f"df_twitterdataset columns: {df_twitterdataset_raw.columns.tolist()}")


### Dropping rows where text or sentiment is null

In [ ]:
df_social_raw = df_sentiment_raw.dropna(subset=['text', 'sentiment'])
df_tweets_raw = df_tweets_raw.dropna(subset=['text', 'sentiment'])
df_twitterdataset_raw = df_twitterdataset_raw.dropna(subset=['text', 'sentiment'])


### Remove Duplicates with in each dataset

In [ ]:

print("\n=== REMOVING DUPLICATES ===")
for name, df in [("df_sentiment_raw", df_sentiment_raw), ("df_tweets_raw", df_tweets_raw), ("df_twitterdataset_raw", df_twitterdataset_raw)]:
    before = len(df)
    df.drop_duplicates(subset=['text'], inplace=True)
    print(f"{name}: {before} -> {len(df)} records")


print("\n=== DUPLICATES CHECK (BEFORE REMOVAL) ===")
print(f"df_sentiment duplicates: {df_sentiment_raw.duplicated(subset='text').sum()}")
print(f"df_tweets duplicates: {df_tweets_raw.duplicated(subset='text').sum()}")
print(f"df_twitterdataset duplicates: {df_twitterdataset_raw.duplicated(subset='text').sum()}")



df_sentiment = df_sentiment_raw.drop_duplicates(subset='text')
df_tweets = df_tweets_raw.drop_duplicates(subset='text')
df_twitterdataset = df_twitterdataset_raw.drop_duplicates(subset='text')


print("\nAfter removing duplicates within each dataset:")
print(f"df_sentiment → {df_sentiment_raw.shape[0]} rows")
print(f"df_tweets → {df_tweets_raw.shape[0]} rows")
print(f"df_twitterdataset → {df_twitterdataset_raw.shape[0]} rows")

### Applying the normalise label function to map the classes 

In [ ]:
df_sentiment_raw['sentiment'] = df_sentiment_raw['sentiment'].apply(normalize_label)
df_tweets_raw['sentiment'] = df_tweets_raw['sentiment'].apply(normalize_label)
df_twitterdataset_raw['sentiment'] = df_twitterdataset_raw['sentiment'].apply(normalize_label)


### Visualising the class distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Keep the same class order and colors across all plots
sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_colors = {
    'Positive': '#4CAF50',
    'Neutral': '#2196F3',
    'Negative': '#F44336'
}
bar_colors = [sentiment_colors[s] for s in sentiment_order]

# Dataset 1: sentimentdataset.csv
df_sentiment['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[0], color=bar_colors, edgecolor='black', width=0.6
)
axes[0].set_title('sentimentdataset.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Number of Rows')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# Dataset 2: Tweets.csv
df_tweets['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[1], color=bar_colors, edgecolor='black', width=0.6
)
axes[1].set_title('Tweets.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment Class')
axes[1].set_ylabel('Number of Rows')
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# Dataset 3: twitterdataset.csv
df_twitterdataset['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[2], color=bar_colors, edgecolor='black', width=0.6
)
axes[2].set_title('twitterdataset.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Sentiment Class')
axes[2].set_ylabel('Number of Rows')
axes[2].tick_params(axis='x', rotation=0)
for p in axes[2].patches:
    axes[2].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.suptitle('Class Distribution in Each Dataset Before Merging', fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig('viz1_distribution_before_merge.png', dpi=150, bbox_inches='tight')
plt.show()

### Dropping rows where label could not be mapped and also viewing what was changed and what was not 

In [ ]:
before_sentiment_r = len(df_sentiment_raw)
before_tweets = len(df_tweets_raw)
before_twitterdataset = len(df_twitterdataset_raw)

 
df_sentiment_raw = df_sentiment.dropna(subset=['sentiment'])
df_tweets_raw = df_tweets.dropna(subset=['sentiment'])
df_twitterdataset_raw = df_twitterdataset.dropna(subset=['sentiment'])
 
print("\n=== LABEL NORMALISATION ===")
print(f"df_sentiment: {before_sentiment_r} → {len(df_sentiment_raw)} rows (dropped {before_sentiment_r - len(df_sentiment_raw)} unmappable labels)")
print(f"df_tweets: {before_tweets} → {len(df_tweets_raw)} rows (dropped {before_tweets - len(df_tweets_raw)} unmappable labels)")
print(f"df_twitterdataset: {before_twitterdataset} → {len(df_twitterdataset_raw)} rows (dropped {before_twitterdataset - len(df_twitterdataset_raw)} unmappable labels)")

print("\ndf_sentiment label distribution after normalisation:")
print(df_sentiment_raw['sentiment'].value_counts())
 
print("\ndf_tweets label distribution after normalisation:")
print(df_tweets_raw['sentiment'].value_counts())

print("\ndf_twitterdataset label distribution after normalisation:")
print(df_twitterdataset_raw['sentiment'].value_counts())
 
 

### Merging the Datasets into one 

In [ ]:
# MERGE THE THREE DATASETS

df_sentiment_raw['source'] = 'social_media'
df_tweets_raw['source'] = 'airline_tweets'
df_twitterdataset_raw['source'] = 'twitterdataset'

combined_df = pd.concat([df_sentiment_raw, df_tweets_raw, df_twitterdataset_raw], ignore_index=True)

print("\n=== AFTER MERGING ===")
print(f"Combined dataset → {combined_df.shape[0]} rows")
print("\nSource breakdown:")
print(combined_df['source'].value_counts())

### Removing cross dataset duplicates

In [ ]:
before_merge_dedup = len(combined_df)
combined_df = combined_df.drop_duplicates(subset='text').reset_index(drop=True)
print(f"\nCross-dataset duplicates removed: {before_merge_dedup - len(combined_df)}")
print(f"Combined dataset after removing duplicates → {combined_df.shape[0]} rows")

# Save a copy BEFORE balancing for later comparison charts
combined_before_balance = combined_df.copy()


### Balancing the Dataset by Reducing Negative Samples

In [ ]:

# Reduce Negative class by 5000 samples in the combined dataset
negative_mask = combined_df['sentiment'] == 'Negative'
negative_count_before = int(negative_mask.sum())
remove_n = min(5000, negative_count_before)

if remove_n > 0:
    drop_idx = combined_df[negative_mask].sample(n=remove_n, random_state=42).index
    combined_df = combined_df.drop(index=drop_idx).reset_index(drop=True)

# Save a copy AFTER balancing
combined_balanced = combined_df.copy()

negative_count_after = int((combined_df['sentiment'] == 'Negative').sum())
print(f"Negative samples removed: {remove_n}")
print(f"Negative count: {negative_count_before} → {negative_count_after}")
print(f"Combined dataset after downsampling → {combined_df.shape[0]} rows")

### Class Imbalance check

In [ ]:
print("\n=== CLASS IMBALANCE CHECK ===")
class_counts = combined_df['sentiment'].value_counts()
print(class_counts)
 
majority = class_counts.max()
minority = class_counts.min()
imbalance_ratio = majority / minority
print(f"\nImbalance ratio (majority / minority): {imbalance_ratio:.2f}x")
 
if imbalance_ratio > 2:
    print("  Imbalance detected — balancing required")
else:
    print(" Classes are reasonably balanced")

### Class Distribution after merging

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Bar chart
colors = ['#4CAF50', '#F44336', '#2196F3']
class_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', width=0.55)
axes[0].set_title('Merged Dataset — Class Distribution (Bar)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)
 
# Pie chart
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Merged Dataset — Class Distribution (Pie)', fontsize=13, fontweight='bold')
 
plt.suptitle('Class Distribution After Merging', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz2_distribution_after_merge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz2_distribution_after_merge.png")
 
 

###   Before vs After balancing visualisation (side by side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_colors = {
    'Positive': '#4CAF50',
    'Neutral': '#2196F3',
    'Negative': '#F44336'
}
bar_colors = [sentiment_colors[s] for s in sentiment_order]

before_counts = combined_before_balance['sentiment'].value_counts().reindex(sentiment_order, fill_value=0)
after_counts = combined_balanced['sentiment'].value_counts().reindex(sentiment_order, fill_value=0)

before_counts.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='black', width=0.55)
axes[0].set_title('Before Balancing', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=11
    )

after_counts.plot(kind='bar', ax=axes[1], color=bar_colors, edgecolor='black', width=0.55)
axes[1].set_title('After Balancing', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=11
    )

plt.suptitle('Class Balancing Effect', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz3_balancing_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz3_balancing_effect.png')

###  Text length distribution by sentiment class

In [ ]:
# This helps us understand whether positive/negative/neutral posts
# tend to be longer or shorter — useful context for the report.
combined_balanced['text_length'] = combined_balanced['text'].str.split().str.len()
 
plt.figure(figsize=(10, 5))
for label, color in zip(['Positive', 'Negative', 'Neutral'], colors):
    subset = combined_balanced[combined_balanced['sentiment'] == label]['text_length']
    sns.kdeplot(subset, label=label, color=color, fill=True, alpha=0.3)
 
plt.title('Text Length Distribution by Sentiment Class', fontsize=13, fontweight='bold')
plt.xlabel('Number of Words')
plt.ylabel('Density')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('viz4_text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz4_text_length_distribution.png")

### Source contribution to final dataset visualisation

In [ ]:
# Shows how much each original dataset contributed after balancing.
 
source_sentiment = combined_balanced.groupby(
    ['source', 'sentiment']
).size().unstack(fill_value=0)
 
source_sentiment.plot(
    kind='bar', color=colors, edgecolor='black', width=0.6, figsize=(9, 5)
)
plt.title('Source Contribution per Sentiment Class', fontsize=13, fontweight='bold')
plt.xlabel('Data Source')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('viz5_source_contribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz5_source_contribution.png")

### Split and Save Training and Validation sets

In [ ]:
final_df = combined_balanced[['text', 'sentiment']].copy()

train_df, val_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=42,
    stratify=final_df['sentiment']
)

train_path = 'data/processed/processed_training_dataset.csv'
val_path = 'data/processed/processed_validation_datset.csv'


print("\n=== FINAL SUMMARY ===")
print(f"Training dataset saved: {len(train_df)} rows")
print(f"Validation dataset saved: {len(val_df)} rows")

print("\nTraining class distribution:")
print(train_df['sentiment'].value_counts())

print("\nValidation class distribution:")
print(val_df['sentiment'].value_counts())

## 5.2 Test Data Preprocessing and Cleaning

### Dropping Duplicates

In [ ]:
df_gmail = df_gmail_raw.drop_duplicates(subset=['text']).reset_index(drop=True)
df_whatsapp = df_whatsapp_raw.drop_duplicates(subset=['text']).reset_index(drop=True)
if 'content' in df_app_reviews_raw.columns:
    df_app_reviews = df_app_reviews_raw.drop_duplicates(subset=['content']).reset_index(drop=True)
else:
    df_app_reviews = df_app_reviews_raw.copy()

print(f"Gmail cleaned: {len(df_gmail)}")
print(f"WhatsApp cleaned: {len(df_whatsapp)}")
print(f"App Reviews cleaned: {len(df_app_reviews)}")

 We define a robust cleaning function using regular expressions to strip signatures, HTML, and specific noise markers identified during EDA.

In [ ]:
#def clean_the_text(text):
#    if not isinstance(text, str): return ""
#    text = re.sub(r'<.*?>', '', text)
#    text = re.sub(r'--- Forwarded message ---', '', text)
#    text = re.sub(r'!!!|\?\?\?|@user', '', text)
#    text = re.sub(r'\s+', ' ', text).strip()
#    return text

def clean_the_text(text):
    # Remove HTML tags (<div>, <br>, etc.)
    text = re.sub(r'<.*?>', '', text)
    
    # Remove Forwarded message markers and signatures
    text = re.sub(r'--- Forwarded message ---', '', text)
    text = re.sub(r'Please consider the environment', '', text, flags=re.I)
    text = re.sub(r'Sent from my (iPhone|mobile|Android|iPhone)', '', text, flags=re.I)
    text = re.sub(r'Best regards, .*', '', text, flags=re.I)
    
    # Remove service markers (MTN:, GitHub:, etc.)
    text = re.sub(r'(MTN|GitHub|Vercel|Udemy|Vultr|CodeMagic|LinkedIn|Amazon|Railway|Netlify|Heroku):', '', text, flags=re.I)

    # Remove specific noise characters (!!!, ???, @user)
    text = re.sub(r'!!!|\?\?\?|@user', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


df_gmail['clean_text'] = df_gmail['text'].apply(clean_the_text)
df_whatsapp['clean_text'] = df_whatsapp['text'].apply(clean_the_text)
df_app_reviews['clean_text'] = df_app_reviews['content'].apply(clean_the_text)

# Note: df_app_reviews['sentiment'] will be provided via manual labeling in Excel

### First five columns of the Data

In [ ]:
print("--- Gmail Preview ---")
print(df_gmail[['text', 'clean_text']].head())
print("--- WhatsApp Preview ---")
print(df_whatsapp[['text', 'clean_text']].head())

print("--- App Reviews Preview ---")
print(df_app_reviews[['clean_text']].head())

### Dropping the columns that are not needed & Saving the datasets

In [ ]:
# Tag sources before merging for visualization
df_gmail['source'] = 'Gmail'
df_whatsapp['source'] = 'WhatsApp'
df_app_reviews['source'] = 'App Reviews'

# Select final columns
gmail_final = df_gmail[['clean_text', 'sentiment', 'source']].rename(columns={'clean_text': 'text'})
whatsapp_final = df_whatsapp[['clean_text', 'sentiment', 'source']].rename(columns={'clean_text': 'text'})
app_reviews_final = df_app_reviews[['clean_text', 'sentiment', 'source']].rename(columns={'clean_text': 'text'})



# saving 
gmail_final.to_csv('/kaggle/working/gmail_test_data.csv', index=False)
whatsapp_final.to_csv('/kaggle/working/whatsapp_test_data.csv', index=False)
app_reviews_final.to_csv('/kaggle/working/labelled_app_reviews_test.csv', index=False)

## 6. Feature Engineering

In [ ]:
def extract_meta_features(df):
    df = df.copy()
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    
    platforms = r'github|slack|coursera|udemy|paystack|railway|netlify|heroku|mtn|airtel|gmail|whatsapp'
    alerts = r'invoice|billing|service termination|payment receipt|account alert|reminder notice|transaction'
    academic = r'assignment|deadline|exam|results|semester|lecture|submission|grade|marks|course'
    
    df['has_platform_mention'] = df['text'].apply(lambda x: 1 if re.search(platforms, str(x).lower()) else 0)
    df['has_service_alert'] = df['text'].apply(lambda x: 1 if re.search(alerts, str(x).lower()) else 0)
    df['exclamation_intensity'] = df['text'].apply(lambda x: min(str(x).count('!'), 5))
    df['technical_success'] = df['text'].apply(lambda x: 1 if re.search(r'successful|approved|passed|accepted|delivered|congratulations|internship|scholarship|live|verified|welcome|registered|great|amazing|won|celebrate', str(x).lower()) else 0)
    df['technical_failure'] = df['text'].apply(lambda x: 1 if re.search(r'failed|rejected|declined|suspended|expired|terminated|error|down|awful|hate|tired|annoying|struggling|frustrated|devastated|embarrassing|breaking', str(x).lower()) else 0)
    df['student_context_score'] = df['text'].apply(lambda x: len(re.findall(academic, str(x).lower())))
    df['positive_signal'] = df['text'].apply(lambda x: len(re.findall(r'congrats|congratulations|proud|excited|happy|amazing|passed|accepted|scholarship|won|celebrate|excellent|well done', str(x).lower())))
    return df

def surgical_cleaner(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = text.translate(str.maketrans('', '', ',.;:()[]{}<>/@#'))
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "notification"

# Ensure test sets have correct column names for features
df_gmail = df_gmail.rename(columns={'clean_text': 'text'})
df_whatsapp = df_whatsapp.rename(columns={'clean_text': 'text'})
if 'clean_text' in df_app_reviews.columns:
    df_app_reviews = df_app_reviews.rename(columns={'clean_text': 'text'})

train_df = extract_meta_features(full_train_df) # Using full training pool
df_gmail = extract_meta_features(df_gmail)
df_whatsapp = extract_meta_features(df_whatsapp)
df_app_reviews = extract_meta_features(df_app_reviews)

train_df['clean'] = train_df['text'].apply(surgical_cleaner)
df_gmail['clean'] = df_gmail['text'].apply(surgical_cleaner)
df_whatsapp['clean'] = df_whatsapp['text'].apply(surgical_cleaner)
df_app_reviews['clean'] = df_app_reviews['text'].apply(surgical_cleaner)

metadata_columns = ['exclamation_count', 'question_count', 'is_all_caps', 'char_cnt', 'word_cnt', 'has_platform_mention', 'has_service_alert', 'exclamation_intensity', 'technical_success', 'technical_failure', 'student_context_score', 'positive_signal']

scaler = StandardScaler() 
X_train_meta = scaler.fit_transform(train_df[metadata_columns]) 
X_gmail_meta = scaler.transform(df_gmail[metadata_columns])
X_whatsapp_meta = scaler.transform(df_whatsapp[metadata_columns])
X_app_meta = scaler.transform(df_app_reviews[metadata_columns])

VOCAB_SIZE, MAX_LEN = 20000, 150
deeplearning_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
deeplearning_tokenizer.fit_on_texts(pd.concat([train_df['clean'], df_gmail['clean'], df_whatsapp['clean'], df_app_reviews['clean']]))

## 7. Ensemble Model Training & Evaluation

In [ ]:
class SentiDS(torch.utils.data.Dataset):
        def __init__(self, enc, lbl): self.enc = enc; self.lbl = lbl
        def __getitem__(self, idx): 
            item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
            item['labels'] = torch.tensor(self.lbl[idx]); return item
        def __len__(self): return len(self.lbl)

def train_distilbert(train_txt, train_lbl):
    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    train_enc = tokenizer(train_txt.tolist(), truncation=True, padding=True, max_length=128)
    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', 
        num_labels=3
        ).to(DEVICE)
    
    args = TrainingArguments(
        output_dir='results', 
        num_train_epochs=3, 
        per_device_train_batch_size=8, 
        gradient_accumulation_steps=4, 
        learning_rate=2e-5, 
        warmup_ratio=0.1, 
        weight_decay=0.01, 
        dataloader_pin_memory=False, 
        fp16=True, 
        disable_tqdm=True, 
        save_strategy='no',
        report_to='none',
        logging_strategy='no')
    
    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=SentiDS(train_enc, train_lbl))
    trainer.train() 
    return model, tokenizer

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

train_predictions = np.zeros((len(train_df), 30)) 
test_preds_gmail = np.zeros((len(df_gmail_test), 30))
test_preds_whatsapp = np.zeros((len(df_whatsapp_test), 30))
test_preds_app = np.zeros((len(df_app_reviews_test), 30))

cw_dict = {i: compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)[i] for i in range(3)} 
all_histories = []; fold_accuracies = []

for fold, (t_idx, v_idx) in enumerate(skf.split(train_df['clean'], y_train)):
    print(f"\n--- WORKING ON FOLD {fold+1} ---")
    
    # TF-IDF Vectorization
    vectoriser = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
    vectoriser.fit(train_df['clean'].iloc[t_idx])
    
    X_t_tfidf = vectoriser.transform(train_df['clean'].iloc[t_idx]) 
    X_v_tfidf = vectoriser.transform(train_df['clean'].iloc[v_idx]) 
    X_test_gmail_tfidf = vectoriser.transform(df_gmail_test['clean'])
    X_test_whatsapp_tfidf = vectoriser.transform(df_whatsapp_test['clean'])
    X_test_app_tfidf = vectoriser.transform(df_app_reviews_test['clean'])
    
    # Random Forest TF-IDF
    vec_rf = TfidfVectorizer(max_features=8000, ngram_range=(1,2), sublinear_tf=True)
    vec_rf.fit(train_df['clean'].iloc[t_idx])
    X_t_rf = vec_rf.transform(train_df['clean'].iloc[t_idx])
    X_v_rf = vec_rf.transform(train_df['clean'].iloc[v_idx])
    X_test_gmail_rf = vec_rf.transform(df_gmail_test['clean'])
    X_test_whatsapp_rf = vec_rf.transform(df_whatsapp_test['clean'])
    X_test_app_rf = vec_rf.transform(df_app_reviews_test['clean'])
    
    # Deep Learning Sequences
    X_t_seq = pad_sequences(deeplearning_tokenizer.texts_to_sequences(train_df['clean'].iloc[t_idx]), maxlen=MAX_LEN)
    X_v_seq = pad_sequences(deeplearning_tokenizer.texts_to_sequences(train_df['clean'].iloc[v_idx]), maxlen=MAX_LEN)
    X_test_gmail_seq = pad_sequences(deeplearning_tokenizer.texts_to_sequences(df_gmail_test['clean']), maxlen=MAX_LEN)
    X_test_whatsapp_seq = pad_sequences(deeplearning_tokenizer.texts_to_sequences(df_whatsapp_test['clean']), maxlen=MAX_LEN)
    X_test_app_seq = pad_sequences(deeplearning_tokenizer.texts_to_sequences(df_app_reviews_test['clean']), maxlen=MAX_LEN)
    
    # --- 1. Naive Bayes ---
    nb = MultinomialNB().fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 0:3] = nb.predict_proba(X_v_tfidf)
    test_preds_gmail[:, 0:3] += nb.predict_proba(X_test_gmail_tfidf) / 5
    test_preds_whatsapp[:, 0:3] += nb.predict_proba(X_test_whatsapp_tfidf) / 5
    test_preds_app[:, 0:3] += nb.predict_proba(X_test_app_tfidf) / 5
    
    # --- 2. Logistic Regression ---
    lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced').fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 3:6] = lr.predict_proba(X_v_tfidf)
    test_preds_gmail[:, 3:6] += lr.predict_proba(X_test_gmail_tfidf) / 5
    test_preds_whatsapp[:, 3:6] += lr.predict_proba(X_test_whatsapp_tfidf) / 5
    test_preds_app[:, 3:6] += lr.predict_proba(X_test_app_tfidf) / 5
    
    # --- 3. SVM ---
    svc = CalibratedClassifierCV(LinearSVC(C=0.5, class_weight='balanced'), cv=3).fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 6:9] = svc.predict_proba(X_v_tfidf)
    test_preds_gmail[:, 6:9] += svc.predict_proba(X_test_gmail_tfidf) / 5
    test_preds_whatsapp[:, 6:9] += svc.predict_proba(X_test_whatsapp_tfidf) / 5
    test_preds_app[:, 6:9] += svc.predict_proba(X_test_app_tfidf) / 5
    
    # --- 4. Random Forest ---
    rf = RandomForestClassifier(n_estimators=500, max_features=0.3, min_samples_leaf=2, class_weight='balanced', n_jobs=-1).fit(X_t_rf, y_train[t_idx])
    train_predictions[v_idx, 9:12] = rf.predict_proba(X_v_rf)
    test_preds_gmail[:, 9:12] += rf.predict_proba(X_test_gmail_rf) / 5
    test_preds_whatsapp[:, 9:12] += rf.predict_proba(X_test_whatsapp_rf) / 5
    test_preds_app[:, 9:12] += rf.predict_proba(X_test_app_rf) / 5
    
    # --- 5. MLP ---
    mlp = MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300, early_stopping=True).fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 12:15] = mlp.predict_proba(X_v_tfidf)
    test_preds_gmail[:, 12:15] += mlp.predict_proba(X_test_gmail_tfidf) / 5
    test_preds_whatsapp[:, 12:15] += mlp.predict_proba(X_test_whatsapp_tfidf) / 5
    test_preds_app[:, 12:15] += mlp.predict_proba(X_test_app_tfidf) / 5
    
    # --- 6. CNN ---
    i_cnn = Input(shape=(MAX_LEN,))
    x_cnn = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(i_cnn)
    x_cnn = SpatialDropout1D(0.3)(x_cnn)
    x_cnn = Conv1D(256, 5, activation='relu', padding='same')(x_cnn)
    cnn_model = Model(inputs=i_cnn, outputs=Dense(3, activation='softmax')(GlobalMaxPooling1D()(x_cnn)))
    cnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    cnn_model.fit(X_t_seq, y_train[t_idx], epochs=5, batch_size=64, verbose=0, class_weight=cw_dict)
    train_predictions[v_idx, 15:18] = cnn_model.predict(X_v_seq)
    test_preds_gmail[:, 15:18] += cnn_model.predict(X_test_gmail_seq) / 5
    test_preds_whatsapp[:, 15:18] += cnn_model.predict(X_test_whatsapp_seq) / 5
    test_preds_app[:, 15:18] += cnn_model.predict(X_test_app_seq) / 5
    
    # --- 7. LSTM ---
    i_lstm = Input(shape=(MAX_LEN,))
    x_lstm = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(i_lstm)
    x_lstm = SpatialDropout1D(0.3)(x_lstm)
    x_lstm = LSTM(128)(x_lstm)
    lstm_model = Model(inputs=i_lstm, outputs=Dense(3, activation='softmax')(x_lstm))
    lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    lstm_model.fit(X_t_seq, y_train[t_idx], epochs=5, batch_size=64, verbose=0, class_weight=cw_dict)
    train_predictions[v_idx, 18:21] = lstm_model.predict(X_v_seq)
    test_preds_gmail[:, 18:21] += lstm_model.predict(X_test_gmail_seq) / 5
    test_preds_whatsapp[:, 18:21] += lstm_model.predict(X_test_whatsapp_seq) / 5
    test_preds_app[:, 18:21] += lstm_model.predict(X_test_app_seq) / 5
    
    # --- 8. Bi-LSTM ---
    i_bi = Input(shape=(MAX_LEN,))
    x_bi = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(i_bi)
    x_bi = SpatialDropout1D(0.3)(x_bi)
    x_bi = Bidirectional(LSTM(64))(x_bi)
    bi_model = Model(inputs=i_bi, outputs=Dense(3, activation='softmax')(x_bi))
    bi_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    bi_model.fit(X_t_seq, y_train[t_idx], epochs=5, batch_size=64, verbose=0, class_weight=cw_dict)
    train_predictions[v_idx, 21:24] = bi_model.predict(X_v_seq)
    test_preds_gmail[:, 21:24] += bi_model.predict(X_test_gmail_seq) / 5
    test_preds_whatsapp[:, 21:24] += bi_model.predict(X_test_whatsapp_seq) / 5
    test_preds_app[:, 21:24] += bi_model.predict(X_test_app_seq) / 5
    
    # --- 9. Bi-GRU ---
    i_gru = Input(shape=(MAX_LEN,))
    x_gru = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(i_gru)
    x_gru = SpatialDropout1D(0.3)(x_gru)
    x_gru = Bidirectional(GRU(64))(x_gru)
    gru_model = Model(inputs=i_gru, outputs=Dense(3, activation='softmax')(x_gru))
    gru_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    gru_model.fit(X_t_seq, y_train[t_idx], epochs=5, batch_size=64, verbose=0, class_weight=cw_dict)
    train_predictions[v_idx, 24:27] = gru_model.predict(X_v_seq)
    test_preds_gmail[:, 24:27] += gru_model.predict(X_test_gmail_seq) / 5
    test_preds_whatsapp[:, 24:27] += gru_model.predict(X_test_whatsapp_seq) / 5
    test_preds_app[:, 24:27] += gru_model.predict(X_test_app_seq) / 5
    
    # --- 10. DistilBERT ---
    bm, bt = train_distilbert(train_df['clean'].iloc[t_idx], y_train[t_idx])
    bm.eval()
    with torch.no_grad():
        def get_p(tl): 
            res = []
            for j in range(0, len(tl), 32): 
                batch = tl[j:j+32]
                e = bt(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
                res.append(torch.softmax(bm(**e).logits, dim=-1).cpu().numpy())
            return np.vstack(res)
        train_predictions[v_idx, 27:30] = get_p(train_df['clean'].iloc[v_idx].tolist())
        test_preds_gmail[:, 27:30] += get_p(df_gmail_test['clean'].tolist()) / 5
        test_preds_whatsapp[:, 27:30] += get_p(df_whatsapp_test['clean'].tolist()) / 5
        test_preds_app[:, 27:30] += get_p(df_app_reviews_test['clean'].tolist()) / 5
    
    print(f"Done Fold {fold+1}!")
    gc.collect(); tf.keras.backend.clear_session(); torch.cuda.empty_cache()

print("\nAll Folds Complete!")

### Stacked Ensemble Meta-Learner (XGBoost)

In [ ]:
X_stack_train = np.hstack([train_predictions, X_train_meta]) 

# Domain-Specific Stacking
X_stack_gmail = np.hstack([test_preds_gmail, X_gmail_meta])
X_stack_whatsapp = np.hstack([test_preds_whatsapp, X_whatsapp_meta])
X_stack_app = np.hstack([test_preds_app, X_app_meta])

final_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, 
    colsample_bytree=0.8, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, 
    objective='multi:softprob', use_label_encoder=False, eval_metric='mlogloss')

final_model.fit(X_stack_train, y_train, verbose=False)

# Final Predictions for All Domains
y_gmail_pred = final_model.predict(X_stack_gmail)
y_whatsapp_pred = final_model.predict(X_stack_whatsapp)
y_app_pred = final_model.predict(X_stack_app)

label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
y_gmail = df_gmail_test['sentiment'].map(label_map).values
y_whatsapp = df_whatsapp_test['sentiment'].map(label_map).values
y_app = df_app_reviews_test['sentiment'].map(label_map).values

## 8. Comparative Analysis & Performance Comparison

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

# ------------------------------------------------------------------
# 0. Training Performance (Source Domain: Twitter)
# ------------------------------------------------------------------
# oof_acc = accuracy_score(y_train, np.argmax(train_predictions, axis=1))
# print(f"Training Accuracy: {oof_acc*100:.2f}%")

# 1. Calculate Individual Domain Accuracies
# [acc_gmail, acc_whatsapp, acc_app calculated here]

# 2. Domain Comparison Visuals
# [Table, Bar Chart, Confusion Matrices]

## 9. Model Interpretability (LIME Stories)

In [ ]:
# [LIME explanation logic here]